# IM2 hillslope routing in a wet step: the `downhill_ti` bug with `spel proptest`

ELM's IM2 hillslope hydrology (`use_IM2_hillslope_hydrology`) passes the surface runoff of each
topounit to the next lower topounit of its gridcell. `HILLSLOPE_IM2.md` (this directory) suspected,
and `HILLSLOPE_IM2_RESULTS.md` confirmed in an ELM run, that `set_topounit` gives the lowest
topounit of some gridcells a downhill neighbour anyway: itself. On the 0.5° surface dataset of the
`topounit_im2` testmod that happens in 7 gridcells; the 5-day January ELM run showed the effect,
but with fractions of a millimetre of runoff.

This notebook makes the steps wet instead. It runs `HydrologyDrainage` (the routine that sends the
water downhill) as a SPEL unit test on synthetic hillslopes, with heavy surface runoff drawn by
Hypothesis, and asks:

1. Which hillslopes get a wrong neighbour, and what does ELM's own code do with them?
2. Do the routing's invariants hold (every sender empties its surface water, routing only moves
   water, the store stays finite, water only flows downhill, mass is conserved)?
3. How much of a gridcell's runoff reaches the river in one wet step, with and without the bug?

It uses the Python API (`spel.proptest.api`) throughout, and writes the spec it builds as
`hillslope_im2.proptest.toml`, so the checks also run from the command line
(`spel proptest run spel/proptest/experiments/hillslope_im2.proptest.toml`).

**What a single step can and cannot show.** The water a topounit sends waits in the downhill
topounit's store (`top_ws%from_uphill`) and is released over the following steps by
`SurfaceRunoff`, in another routine (`HydrologyNoDrainage`). One call of `HydrologyDrainage` shows
where each step's runoff goes; the season, with the store filling and draining, needs multi-step
runs, which proptest does not have yet (see the end).

## 0. Setup

The case is `hs_bug` (made by `spel validate` from the `next` worktree of E3SM master, see
`hillslope_validate.txt`); its reference capture holds ELM's state in the 7 gridcells with a
self-looping outlet. `SPEL_E3SM_SRCROOT` points SPEL at that checkout before anything imports
`spel.scripts.config`; the library is built with the machine's gfortran toolchain.

In [1]:
import os
os.environ.setdefault("SPEL_E3SM_SRCROOT", "/lcrc/globalscratch/ac.schwartzpd/next")

import copy
import io
from collections import Counter
from pathlib import Path

import netCDF4
import numpy as np
import pandas as pd

from spel.proptest import api as pt, runner
from spel.proptest.api import Gridcell, Soil, Topounit
from spel.proptest.hierarchy import Layout

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)

HERE = Path.cwd()                                 # spel/proptest/experiments
SRCROOT = Path(os.environ["SPEL_E3SM_SRCROOT"])
CASE, REFERENCE = "hs_bug", "hs_bug"

## 1. The bug, in ELM's words and in Python

`set_topounit` (initGridCellsMod) looks for each topounit's highest lower neighbour. `dn_index`
is set to -1 once, *before* the loop over topounits:

In [2]:
src = (SRCROOT / "components/elm/src/main/initGridCellsMod.F90").read_text().splitlines()
k = next(i for i, l in enumerate(src) if "find the closest downhill neighbor" in l)
print("\n".join(f"{k + 1 + j:5d}  {l}" for j, l in enumerate(src[k:k + 13])))

  315        ! find the closest downhill neighbor for each topounit
  316        dn_index = -1  ! value of -1 indicates no downhill neighbor
  317        do t1 = begt, endt
  318           t1_elev = top_pp%elevation(t1)
  319           dn_elev = min_elev
  320           do t2 = begt, endt
  321              t2_elev = top_pp%elevation(t2)
  322              if ((t2_elev < t1_elev) .and. (t2_elev >= dn_elev)) then
  323                 dn_elev = t2_elev
  324                 dn_index = t2
  325              endif
  326           end do
  327           top_pp%downhill_ti(t1)=dn_index


A topounit with no lower neighbour (the lowest one) keeps the previous topounit's `dn_index`. The
same loop in Python, next to the intended one (`dn_index` reset for every topounit), on the
hillslopes this notebook uses. Each is a list of `(elevation in m, weight)` in the order the
surface data lists the topounits; two are real gridcells of the testmod's dataset
(`HILLSLOPE_IM2_RESULTS.md`, Phase 1; weights rounded to sum to 1):

In [3]:
HILLS = {
    "lowest first":   [(50, .3), (150, .3), (300, .4)],
    "lowest last":    [(300, .4), (150, .3), (50, .3)],   # the same hillslope as "lowest first", reversed
    "valley":         [(100, .3), (50, .3), (300, .4)],
    "cycle":          [(300, .25), (200, .25), (50, .25), (150, .25)],
    "tie":            [(100, .4), (50, .3), (50, .3)],
    "caspian 184059": [(1.8, .70), (-20.7, .15), (-64.9, .10), (-110.7, .05)],
    "salton 177249":  [(14.1, .9), (-50.7, .05), (-163.0, .05)],
    "inactive low":   [(300, .5), (150, .5), (50, 0.0)],  # weight 0: an inactive topounit (H2)
}


def elm_downhill(elev):
    # set_topounit's loop, 0-based: dn_index is set once, before the loop
    lo, dn, out = min(elev), -1, []
    for e1 in elev:
        de = lo
        for j, e2 in enumerate(elev):
            if e2 < e1 and e2 >= de:
                de, dn = e2, j
        out.append(dn)
    return out


def intended_downhill(elev):
    out = []
    for e1 in elev:
        lower = [j for j, e2 in enumerate(elev) if e2 < e1]
        out.append(max(lower, key=lambda j: (elev[j], j)) if lower else -1)
    return out


def verdict(elev, dn):
    bad = []
    for i, j in enumerate(dn):
        if j == i:
            bad.append(f"{i + 1} -> itself")
        elif j >= 0 and elev[j] >= elev[i]:
            bad.append(f"{i + 1} -> {j + 1} ({'level' if elev[j] == elev[i] else 'uphill'})")
    return ", ".join(bad) or "ok"


rows = []
for name, tops in HILLS.items():
    elev = [e for e, _ in tops]
    elm, want = elm_downhill(elev), intended_downhill(elev)
    one = lambda d: [x + 1 if x >= 0 else -1 for x in d]  # 1-based within the gridcell, -1: none
    rows.append(dict(hillslope=name, elevations=elev, elm=one(elm), intended=one(want), problem=verdict(elev, elm)))
pd.DataFrame(rows).set_index("hillslope")

,elevations,elm,intended,problem
hillslope,,,,
lowest first,"[50, 150, 300]","[-1, 1, 2]","[-1, 1, 2]",ok
lowest last,"[300, 150, 50]","[2, 3, 3]","[2, 3, -1]",3 -> itself
valley,"[100, 50, 300]","[2, 2, 1]","[2, -1, 1]",2 -> itself
cycle,"[300, 200, 50, 150]","[2, 4, 4, 3]","[2, 4, -1, 3]",3 -> 4 (uphill)
tie,"[100, 50, 50]","[3, 3, 3]","[3, -1, -1]","2 -> 3 (level), 3 -> itself"
caspian 184059,"[1.8, -20.7, -64.9, -110.7]","[2, 3, 4, 4]","[2, 3, 4, -1]",4 -> itself
salton 177249,"[14.1, -50.7, -163.0]","[2, 3, 3]","[2, 3, -1]",3 -> itself
inactive low,"[300, 150, 50]","[2, 3, 3]","[2, 3, -1]",3 -> itself


Only the hillslope stored lowest first is right. The real gridcells are listed highest first, so
their outlet points at itself; "cycle" sends its lowest topounit's water uphill into a loop;
"tie" points topounits at an equal one. "inactive low" adds the second suspected problem (H2):
the search ignores activity, so the middle topounit sends to a topounit of weight 0, and the send
divides by that weight.

ELM itself did the same in the `hs-im2` run. The reference capture of `hs_bug` holds ELM's
`top_pp%downhill_ti` for those 7 gridcells (merged from 7 MPI ranks; indices renumbered from 1):

In [4]:
ref = netCDF4.Dataset(Path(pt.Case(CASE, reference=REFERENCE).reference) / "spel-inputs0001.nc")
ref.set_auto_mask(False)
dn = np.asarray(ref["top_pp__downhill_ti"][0])
k = np.asarray(ref["top_pp__topo_grc_ind"][0])
w = np.asarray(ref["top_pp__wtgcell"][0])
g = np.cumsum(k == 1)                                   # topo_grc_ind restarts at 1 in each gridcell
cap = pd.DataFrame({"gindex": np.asarray(ref.spel_subset_gindex)[g - 1], "lat": np.asarray(ref.spel_subset_latdeg)[g - 1],
                    "lon": np.asarray(ref.spel_subset_londeg)[g - 1], "topounit": np.arange(1, len(dn) + 1),
                    "number in gridcell": k, "weight": w, "downhill_ti": dn})
cap["points at itself"] = cap["downhill_ti"] == cap["topounit"]
ref.close()
cap

,gindex,lat,lon,topounit,number in gridcell,weight,downhill_ti,points at itself
0,177249,33.25,-115.75,1,1,0.9002,2,False
1,177249,33.25,-115.75,2,2,0.0499,3,False
2,177249,33.25,-115.75,3,3,0.0500,3,True
3,184059,37.75,49.25,4,1,0.7007,5,False
4,184059,37.75,49.25,5,2,0.1497,6,False
5,184059,37.75,49.25,6,3,0.0996,7,False
6,184059,37.75,49.25,7,4,0.0500,7,True
7,186937,39.75,48.25,8,1,0.9506,9,False
8,186937,39.75,48.25,9,2,0.0494,9,True
9,187666,40.25,52.75,10,1,0.9501,11,False


## 2. The case

`pt.Case` loads the unit-test case's inventory: every variable `HydrologyDrainage` touches,
classified by how the routine uses it. `use_im2_hillslope_hydrology` is one of the namelist
options it tests. The IM2 state is inout (`top_ws%from_uphill`, which the routine adds to, and
`col_wf%qflx_to_downhill`, which it sets only in columns with a neighbour), and
`top_pp%downhill_ti` is topology, which the synthetic domain computes from the declared
elevations.

In [5]:
case = pt.Case(CASE, reference=REFERENCE)
print("routines:", case.inv.routines)
print(dict(Counter(v.kind for v in case.inv.vars.values())))
pd.DataFrame([dict(name=v.name, kind=v.kind, dims=v.dims, units=v.units) for v in
              case.find("downhill") + case.find("uphill") + case.find("qflx_surf", n=3)]).drop_duplicates("name")

routines: ['hydrologydrainagemod::hydrologydrainage']
{'unused': 18, 'parameter': 7, 'topology': 13, 'output': 13, 'inout': 22, 'input': 37}


,name,kind,dims,units
0,top_pp%downhill_ti,topology,"(begt:endt,)",
1,col_wf%qflx_to_downhill,inout,"(begc:endc,)",mm/s
2,top_ws%from_uphill,inout,"(begt:endt,)",mm
5,col_wf%qflx_surf,inout,"(begc:endc,)",mm/s
6,col_wf%qflx_h2osfc_surf,inout,"(begc:endc,)",mm/s
7,col_wf%qflx_runoff,output,"(begc:endc,)",mm/s


## 3. The domain: synthetic hillslopes

Each hillslope becomes a gridcell of soil topounits with the declared elevations and weights
(`hierarchy.Gridcell`/`Topounit`; one soil column with a C3 grass patch per topounit). Two sets,
as relations need. The domain computes `top_pp%downhill_ti` with `set_topounit`'s algorithm,
quirk included, so it must agree with the Python transcription above:

In [6]:
def hill(name, tops):
    return Gridcell(name=name, topounits=tuple(Topounit((Soil(13),), weight=w, elevation=e) for e, w in tops))


dom = pt.Domain([hill(n, t) for n, t in HILLS.items()], nsets=2, reference=REFERENCE)
lay = Layout(dom.gridcells, dom.nsets)
tt = lay.table("t")
tt["elevation"] = [HILLS[n][j - 1][0] for n, j in zip(tt["name"], tt["topo_number"])]
tt["downhill_ti"] = lay.downhill()

# the domain's neighbours = elm_downhill, gridcell by gridcell (global indices -> number in the gridcell)
first = tt.groupby("gridcell")["topounit"].transform("min")
local = np.where(tt["downhill_ti"] > 0, tt["downhill_ti"] - first + 1, -1)
expect = np.concatenate([[x + 1 if x >= 0 else -1 for x in elm_downhill([e for e, _ in HILLS[n]])]
                         for n in lay.table("g")["name"]])
assert (local == expect).all()
tt[tt["set"] == 0][["name", "topo_number", "weight", "active", "elevation", "downhill_ti"]]

,name,topo_number,weight,active,elevation,downhill_ti
topounit,,,,,,
1,lowest first,1,0.30,True,50.0,-1
2,lowest first,2,0.30,True,150.0,1
3,lowest first,3,0.40,True,300.0,2
4,lowest last,1,0.40,True,300.0,5
5,lowest last,2,0.30,True,150.0,6
6,lowest last,3,0.30,True,50.0,6
7,valley,1,0.30,True,100.0,8
8,valley,2,0.30,True,50.0,8
9,valley,3,0.40,True,300.0,7


## 4. A wet scenario

Every input the routine reads needs a rule. `case.stats_rules` writes a rule file from the
reference capture (1st–99th percentiles of each input, under the scenario's namelist options);
the scenario extends it and replaces a few rules:

* **wet**: surface runoff `qflx_surf` log-uniform from 1e-6 to 2e-3 mm/s (up to about 7 mm/h),
  surface-water runoff `qflx_h2osfc_surf` up to 1e-4 mm/s, rain up to 5e-3 mm/s, and up to
  5 kg/m² already waiting in each store from earlier steps. The January capture alone would draw
  fractions of a millimetre;
* **soil geometry**: `col_pp%dz`, `z`, `zi` take ELM's vertical grid, the same in every soil column
  (one value per level, from the capture). Drawn independently per level, as the statistics
  would, layer depths and interfaces do not fit together, and while this notebook was written
  Hypothesis found examples where that made the drainage (`qflx_runoff`) NaN in every column;
* `col_pp%nlevbed` (depth to bedrock as a layer index) is a ruled input; the reference has 10.

The checks run over `c in nolakec`, the columns `HydrologyDrainage` loops over: inactive columns
are not in it, so their routing variables keep whatever they held.

In [7]:
def reference_geometry(case):
    # ELM's soil vertical grid, from the first active soil column of the capture: a value per level
    out = {}
    with netCDF4.Dataset(case.reference / "spel-inputs0001.nc") as d:
        d.set_auto_mask(False)
        active = np.asarray(d["col_pp__active"][0]).astype(bool)
        col = int(np.nonzero(active & (np.asarray(d["col_pp__itype"][0]) == 1))[0][0])
        for name in ("col_pp%dz", "col_pp%z", "col_pp%zi"):
            v = d[name.replace("%", "__")]
            vals, lb = np.asarray(v[0])[:, col], int(np.atleast_1d(v.lbounds)[-1])
            out[name] = pt.Value(0.0, levels={str(lb + j): pt.Value(float(x)) for j, x in enumerate(vals)})
    return out


IM2 = {"use_IM2_hillslope_hydrology": True}
spec = pt.Spec(case, dom, namelist=IM2, build={"fc": "gfortran"}, max_examples=40,
               path=HERE / "hillslope_im2.proptest.toml")
rules = case.stats_rules(namelist=IM2)
wet = spec.scenario("wet", over="c in nolakec", extends=rules, inputs={
    "col_wf%qflx_surf": pt.LogRange(1e-6, 2e-3),
    "col_wf%qflx_h2osfc_surf": pt.LogRange(1e-8, 1e-4),
    "top_af%rain": pt.Range(0.0, 5e-3),
    "top_ws%from_uphill": pt.Range(0.0, 5.0),
    **reference_geometry(case),
}, comment="heavy surface runoff on ELM's soil grid; the rest from the hs_bug capture")
print(rules)

/gpfs/fs1/home/ac.schwartzpd/SPEL/unit-tests/proptest-data/hs_bug/stats/hs_bug.inputs.toml


## 5. The checks

Written as Fortran expressions over the elements (`c`, and `t` for its topounit), so they go into
the TOML spec. The first two state what the IM2 code must do whatever the topology; the next two
are the bug's signatures; the relation runs the same example with IM2 off:

| check | expected |
|---|---|
| a topounit with a downhill neighbour keeps no positive surface water (`frac_to_downhill` = 1) | holds |
| no topounit sends water to itself | fails: H1 |
| the downhill store stays finite | fails: H2 |
| IM2 off: a column's runoff is its IM2-on runoff plus what it sent downhill | holds: routing only moves water |

In [8]:
spec.property("topounits with a downhill neighbour keep no positive surface water",
              "implies(top_pp%downhill_ti(t) /= -1, col_wf%qflx_surf(c) <= 0.0 .and. col_wf%qflx_h2osfc_surf(c) <= 0.0)")
spec.property("no topounit sends water to itself",
              "implies(col_wf%qflx_to_downhill(c) > 0.0, top_pp%downhill_ti(t) /= t)")
spec.property("the downhill store stays finite", "finite(top_ws%from_uphill(t))", over="t")
spec.relation("IM2 off: the runoff a column sent downhill stays in its own runoff",
              "close(follow%col_wf%qflx_runoff(c), base%col_wf%qflx_runoff(c) + base%col_wf%qflx_to_downhill(c),"
              " rtol=1.0e-12, atol=1.0e-25)",
              namelist={"use_IM2_hillslope_hydrology": False})
spec.to_toml(spec.path)
print(spec.path.name, "written;", len(spec.path.read_text().splitlines()), "lines")

hillslope_im2.proptest.toml written; 113 lines


## 6. One wet step, by hand

`spec.session()` builds the library if needed and starts the worker. `built().check()` compares
what the case built with the declaration (empty: topology, bounds and filters agree). Then one
example, drawn like a test run's, for the mirror pair "lowest first" / "lowest last":

In [9]:
s = spec.session()
print("built domain vs declaration:", s.built().check() or "no differences")
DN = np.asarray(s.static("top_pp%downhill_ti").data)
res = s.execute("wet", s.sample("wet", seed=3))
cf = res.frame("c", names=["col_wf%qflx_surf", "col_wf%qflx_to_downhill", "col_wf%qflx_runoff"])
cf["elevation"] = tt["elevation"].to_numpy()[lay.ancestor_of("c", "t")]
cf["downhill_ti"] = DN[lay.ancestor_of("c", "t")]
cf["qflx_surf before"] = np.asarray(res.example.inputs["col_wf%qflx_surf"])
cols = ["name", "topounit", "elevation", "downhill_ti", "qflx_surf before", "col_wf%qflx_surf",
        "col_wf%qflx_to_downhill", "col_wf%qflx_runoff"]
cf[(cf["set"] == 0) & cf["name"].isin(["lowest first", "lowest last"])][cols]

spel proptest: building /gpfs/fs1/home/ac.schwartzpd/SPEL/unit-tests/hs_bug/build-py/libspelcase.so (no library yet)


machine chrysalis: toolchain gfortran (SPEL_FC overrides; `spel machine env --fc gfortran` shows it)


built domain vs declaration: no differences


,name,topounit,elevation,downhill_ti,qflx_surf before,col_wf%qflx_surf,col_wf%qflx_to_downhill,col_wf%qflx_runoff
column,,,,,,,,
1,lowest first,1,50.0,-1,5.5422e-05,5.5422e-05,0.0000e+00,5.6016e-05
2,lowest first,2,150.0,1,5.4902e-04,0.0000e+00,5.4956e-04,4.0297e-08
3,lowest first,3,300.0,2,6.0057e-05,0.0000e+00,6.0597e-05,9.4986e-08
4,lowest last,4,300.0,5,3.2378e-04,0.0000e+00,3.2433e-04,5.9291e-05
5,lowest last,5,150.0,6,1.5541e-04,0.0000e+00,1.5595e-04,4.0696e-02
6,lowest last,6,50.0,6,1.1134e-03,0.0000e+00,1.1140e-03,4.0102e-09


In "lowest first" the outlet (elevation 50, no neighbour) keeps its surface runoff, and it is in
its `qflx_runoff`, which goes to the river. The two topounits above it send theirs downhill. In
"lowest last" the outlet's `downhill_ti` is itself: its surface runoff goes to its own store, and
nothing of the hillslope's surface runoff is left in any column. (`qflx_runoff` also holds the
subsurface drainage, which IM2 does not route; the inputs of the two hillslopes are drawn
independently here, so their drainage differs.)

## 7. Running the checks

`runner.run` gives every check a Hypothesis test (40 examples, seed 0) and returns an `Outcome`
each; a failure is shrunk to a minimal example and saved under
`unit-tests/proptest-data/hs_bug/failures/`.

In [10]:
log = io.StringIO()
outcomes = runner.run(spec.validate(), case.inv, settings={"seed": 0}, session=s, out=log)
pd.DataFrame([(o.check, o.kind, "pass" if o.passed else "FAIL") for o in outcomes],
             columns=["check", "kind", "result"])

,check,kind,result
0,topounits with a downhill neighbour keep no po...,property,pass
1,no topounit sends water to itself,property,FAIL
2,the downhill store stays finite,property,FAIL
3,IM2 off: the runoff a column sent downhill sta...,relation,pass


In [11]:
for o in outcomes:
    if not o.passed:
        lines = o.message.splitlines()
        print("\n".join(lines[:6]))
        print("   ...\n")

no topounit sends water to itself: implies(col_wf%qflx_to_downhill(c) > 0.0, top_pp%downhill_ti(t) /= t)
  fails at 10 of 50 elements; first: element 6 (set 0, lowest last, topounit 3/3, soil, column soil)
    implies = False
    col_wf%qflx_to_downhill = 3.1225107803338285e-05
    top_pp%downhill_ti = 6
  drawn inputs at that element:
   ...

the downhill store stays finite: finite(top_ws%from_uphill(t))
  fails at 2 of 52 elements; first: element 26 (set 0, inactive low, topounit 3/3)
    finite = False
    top_ws%from_uphill = inf
  drawn inputs at that element:
    atm2lnd_vars%forc_flood_grc = 0.0
   ...



The two invariants hold; the two signatures fail where expected: the self-loops at 10 of the 50
active columns (the five self-looping outlets of the table in part 1, in both sets), and the
store at the weight-0 topounit of "inactive low" (H2) in both sets, where it is `Inf`. Each
failure is shrunk and saved for `spel proptest replay`. The relation holds in every active column:
routing only moves runoff between columns, so the column water balance closes and ELM's balance
checks cannot see either bug.

## 8. Checks in Python: direction and mass

Two properties need what the case does not hold: the topounits' elevations (only the
domain declaration knows them) and sums over a gridcell. `Result.frame` gives the state per
element with the layout's description, and the layout gives weights and ancestors.

**Water flows downhill**: every active topounit that sent water has a lower target.

In [12]:
active_t = lay.active("t")
elev = tt["elevation"].to_numpy()
fails = []
for seed in range(10):
    r = s.execute("wet", s.sample("wet", seed=seed))
    tf = r.frame("t", names=[], aggregate={"col_wf%qflx_to_downhill": "wsum"})
    sent = (tf["wsum(col_wf%qflx_to_downhill)"].to_numpy() > 0) & active_t
    target = np.where(DN > 0, elev[np.clip(DN - 1, 0, None)], np.nan)
    wrong = sent & ~(target < elev)
    fails += [dict(seed=seed, set=tt["set"].iloc[i], hillslope=tt["name"].iloc[i], topounit=i + 1,
                   elevation=elev[i], downhill_ti=DN[i], target_elevation=target[i]) for i in np.nonzero(wrong)[0]]
fails = pd.DataFrame(fails)
fails[fails["seed"] == 0]

,seed,set,hillslope,topounit,elevation,downhill_ti,target_elevation
0,0,0,lowest last,6,50.0,6,50.0
1,0,0,valley,8,50.0,8,50.0
2,0,0,cycle,12,50.0,13,150.0
3,0,0,tie,15,50.0,16,50.0
4,0,0,tie,16,50.0,16,50.0
5,0,0,caspian 184059,20,-110.7,20,-110.7
6,0,0,salton 177249,23,-163.0,23,-163.0
7,0,1,lowest last,32,50.0,32,50.0
8,0,1,valley,34,50.0,34,50.0
9,0,1,cycle,38,50.0,39,150.0


In [13]:
fails.groupby("hillslope").size().rename("sends that are not downhill, 10 examples x 2 sets")

hillslope
caspian 184059    20
cycle             20
lowest last       20
salton 177249     20
tie               40
valley            20
Name: sends that are not downhill, 10 examples x 2 sets, dtype: int64

**Mass**: per gridcell, the water its active columns sent (kg/m² of gridcell) equals the water its
topounits' stores gained (weighted by the topounits' weights), up to rounding.

In [14]:
DTIME = 1800.0
gt = lay.table("g")
w_c, w_t = lay.weight("c", "g"), lay.weight("t", "g")
g_of_c, g_of_t = lay.ancestor_of("c", "g"), lay.ancestor_of("t", "g")
active_c = lay.active("c")
rows = []
for seed in range(10):
    r = s.execute("wet", s.sample("wet", seed=seed))
    sent = np.where(active_c, np.asarray(r.arrays["col_wf%qflx_to_downhill"].data), 0.0) * DTIME * w_c
    store, store0 = np.asarray(r.arrays["top_ws%from_uphill"].data), np.asarray(r.example.inputs["top_ws%from_uphill"])
    gain = np.where(np.isfinite(store), store - store0, 0.0) * w_t
    sent_g, gain_g = np.bincount(g_of_c, sent, lay.count("g")), np.bincount(g_of_t, gain, lay.count("g"))
    rows += [dict(seed=seed, hillslope=gt["name"].iloc[i], sent=sent_g[i], stored=gain_g[i]) for i in range(len(gt))]
mass = pd.DataFrame(rows)
mass["lost"] = mass["sent"] - mass["stored"]
mass.groupby("hillslope").agg(max_sent=("sent", "max"), max_abs_lost=("lost", lambda x: np.abs(x).max()))

,max_sent,max_abs_lost
hillslope,,
caspian 184059,3.0925,4.4409e-16
cycle,3.6322,1.1102e-16
inactive low,1.8909,1.8900e+00
lowest first,2.5255,4.4409e-16
lowest last,3.6322,2.2204e-16
salton 177249,3.7800,4.4409e-16
tie,2.1930,4.4409e-16
valley,3.6011,2.2204e-16


Mass is conserved to rounding everywhere except "inactive low", where the water sent to the
weight-0 topounit becomes an `Inf` in its store: lost to the water budget (and an `Inf` in the
restart file). The self-loops conserve mass; their problem is where the water goes.

## 9. How much reaches the river in a wet step?

"lowest first" and "lowest last" are the same hillslope stored in opposite orders. To compare them
fairly, each example is changed so that every element of "lowest last" gets the drawn inputs of
the element of "lowest first" at the same elevation (same set). Then:

* with IM2 **off** (a sibling session under the other namelist option) the two must give the same
  results up to rounding (their columns are summed in the opposite order), which checks the
  mirroring;
* with IM2 **on**, the per-gridcell runoff of the routed kinds (surface, surface-water, perched)
  left in the columns, which goes to the river this step, and what was sent to the stores.

In [15]:
def mirror(ex, src, dst):
    # a copy of `ex` in which hillslope dst has the inputs of src, element by element of the same elevation
    out = copy.deepcopy(ex)
    ct = lay.table("c")
    maps = {}
    for level, table in (("t", tt), ("c", ct)):
        e = elev if level == "t" else elev[lay.ancestor_of("c", "t")]
        idx = np.arange(len(table))
        m = {}
        for k in sorted(set(table["set"])):
            a = idx[(table["name"] == src).to_numpy() & (table["set"] == k).to_numpy()]
            b = idx[(table["name"] == dst).to_numpy() & (table["set"] == k).to_numpy()]
            m.update({j: a[np.nonzero(e[a] == e[j])[0][0]] for j in b})
        maps[level] = m
    for name, arr in out.inputs.items():
        var = case.inv.vars.get(name)
        if var is None or var.level not in maps:
            continue
        x = np.moveaxis(np.asarray(arr), var.subgrid_axis, 0)
        for j, i in maps[var.level].items():
            x[j] = x[i]
        out.inputs[name] = np.moveaxis(x, 0, var.subgrid_axis)
    return out


ROUTED = ("col_wf%qflx_surf", "col_wf%qflx_h2osfc_surf", "col_wf%qflx_drain_perched")


def budget(r):
    # per gridcell, kg/m2 of gridcell in the step: routed-kind runoff left in the columns, and sent downhill
    left = sum(np.nan_to_num(np.asarray(r.arrays[n].data)) for n in ROUTED)
    sent = np.nan_to_num(np.asarray(r.arrays["col_wf%qflx_to_downhill"].data)) if "col_wf%qflx_to_downhill" in r.arrays else 0 * left
    w = w_c * active_c * DTIME
    return np.bincount(g_of_c, left * w, lay.count("g")), np.bincount(g_of_c, sent * w, lay.count("g"))


off = s.sibling({"use_im2_hillslope_hydrology": False})
twins = gt["name"].isin(["lowest first", "lowest last"]).to_numpy()
rows = []
for seed in range(40):
    ex = mirror(s.sample("wet", seed=seed), "lowest first", "lowest last")
    river_on, sent_on = budget(s.execute("wet", ex))
    river_off, _ = budget(off.execute("wet", ex))
    rows += [dict(seed=seed, set=gt["set"].iloc[i], hillslope=gt["name"].iloc[i], river_off=river_off[i],
                  river_on=river_on[i], sent_on=sent_on[i]) for i in np.nonzero(twins)[0]]
tw = pd.DataFrame(rows).pivot_table(index=["seed", "set"], columns="hillslope")
a, b = tw["river_off"]["lowest first"].to_numpy(), tw["river_off"]["lowest last"].to_numpy()
print("IM2 off, largest relative difference between the twins:", np.max(np.abs(a - b) / np.abs(a)))
tw.head(6)

IM2 off, largest relative difference between the twins: 2.0610021561539778e-16


river_off                 river_on                  sent_on            
hillslope lowest first lowest last lowest first lowest last lowest first lowest last
seed set                                                                            
0    0          2.5528      2.5528       1.0897         0.0       1.4631      2.5528
     1          0.0340      0.0340       0.0102         0.0       0.0238      0.0340
1    0          0.1525      0.1525       0.0458         0.0       0.1068      0.1525
     1          0.1525      0.1525       0.0458         0.0       0.1068      0.1525
2    0          1.1129      1.1129       1.0807         0.0       0.0323      1.1129
     1          3.6083      3.6083       1.0829         0.0       2.5255      3.6083

In [16]:
summary = pd.DataFrame({
    "to the river (IM2 off)": tw["river_off"]["lowest first"],
    "to the river, intended neighbours": tw["river_on"]["lowest first"],
    "to the river, the bug": tw["river_on"]["lowest last"],
    "to the stores, intended": tw["sent_on"]["lowest first"],
    "to the stores, the bug": tw["sent_on"]["lowest last"],
})
summary.describe().loc[["count", "mean", "50%", "max"]]

,to the river (IM2 off),"to the river, intended neighbours","to the river, the bug","to the stores, intended","to the stores, the bug"
count,80.0000,80.0000,80.0,80.0000,80.0000
mean,1.0032,0.3261,0.0,0.6772,1.0032
50%,0.6920,0.0557,0.0,0.3313,0.6920
max,3.6513,1.1340,0.0,2.5686,3.6513


IM2 off, the twins agree to rounding: the mirroring works. Over the 80 twin pairs (40 examples,
2 sets) a hillslope produces on average about 1.0 kg/m² of routed-kind runoff in the step. With the
intended neighbours about a third of it (mean 0.33, up to 1.13 kg/m²) reaches the river in the
same step: the outlet's own share (its weight is 0.3 of the gridcell); the rest waits in the
downhill stores. With the bug **none** reaches the river from these hillslopes, in any of the 80
pairs: all of it,
including the outlet's own, goes into stores, and the outlet's store releases into the outlet's
own soil. In the real gridcells (the Caspian one's outlet is 5% of the gridcell, the Salton Sea
one's 5% too) the whole gridcell's surface runoff converges on that small outlet.

## 10. What this shows, and what it does not

* ELM's neighbour search gives self-loops, uphill pointers and loops depending only on the order
  of the topounits in the surface data; a weight-0 target turns the store into `Inf` (H2).
* Under a wet step, the IM2 invariants that do not depend on the topology hold: every sender
  empties its surface water, routing only moves water between columns, and mass is conserved
  (except H2). This is why nothing in ELM complains.
* On a self-looping hillslope none of the routed surface runoff reaches the river in the step.
  In "lowest first" the outlet passes its own share on at once.

Not shown: what happens over a season. The store releases `frac_from_uphill` (0.5) of its water
per step into the soil of the receiving topounit (`SurfaceRunoff`, in `HydrologyNoDrainage`),
where it infiltrates, evaporates, or runs off again and goes back to a store. Measuring the
delayed and lost runoff needs a case with both routines and multi-step runs that carry the state
from step to step; that is the next step for proptest (`HILLSLOPE_IM2.md`, Phase 3, item 3).

In [17]:
s.close()